In [1]:
import os
import joblib
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "cleaned_dataset.csv")
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

df = pd.read_csv(PROCESSED_DATA_PATH)
print("Master cleaned dataset loaded.")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Master cleaned dataset loaded.
Shape: 7308 rows, 11 columns


In [2]:
linear_features = [
    "WaterLevel_m",
    "SoilMoisture_pct",
    "Location_Marikina",
    "Location_Pasig",
    "Location_Quezon City"
]
cart_features = [
    "WaterLevel_m",
    "SoilMoisture_pct",
    "Elevation_m",
    "Location_Manila",
    "Location_Marikina",
    "Location_Pasig",
    "Location_Quezon City"
]

y_reg = df["Rainfall_mm"]
y_clf = df["FloodOccurrence"]

# regression models
lin_reg = LinearRegression().fit(df[linear_features], y_reg)
cart_reg = DecisionTreeRegressor(max_depth=4, min_samples_leaf=20, random_state=42).fit(df[cart_features], y_reg)

# classification models (practice)
log_clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42).fit(df[linear_features], y_clf)
cart_clf = DecisionTreeClassifier(max_depth=4, class_weight="balanced", min_samples_leaf=20, random_state=42).fit(df[cart_features], y_clf)

artifacts = {
    "baseline_linear_regression.joblib": lin_reg,
    "cart_decision_tree_regressor.joblib": cart_reg,
    "baseline_logistic_regression.joblib": log_clf,
    "cart_decision_tree_classifier.joblib": cart_clf
}

for filename, model_obj in artifacts.items():
    file_path = os.path.join(MODELS_DIR, filename)
    joblib.dump(model_obj, file_path)
    print(f"Exported: {file_path}")

# export feature metadata for dashboard input
feature_metadata = {
    "linear_features": linear_features,
    "cart_features": cart_features
}
metadata_path = os.path.join(MODELS_DIR, "model_features.json")
with open(metadata_path, "w") as f:
    json.dump(feature_metadata, f, indent=4)

print(f"\nSaved feature schema metadata to: {metadata_path}")

Exported: ..\models\baseline_linear_regression.joblib
Exported: ..\models\cart_decision_tree_regressor.joblib
Exported: ..\models\baseline_logistic_regression.joblib
Exported: ..\models\cart_decision_tree_classifier.joblib

Saved feature schema metadata to: ..\models\model_features.json


In [3]:
# TASK 4.2: DESERIALIZATION & INFERENCE VERIFICATION
# Simulates the production environment (e.g., Streamlit/FastAPI backend)

# 1. Load feature schemas
with open(os.path.join(MODELS_DIR, "model_features.json"), "r") as f:
    saved_features = json.load(f)

# 2. Re-load trained model artifacts from disk
loaded_lin_reg = joblib.load(
    os.path.join(MODELS_DIR, "baseline_linear_regression.joblib")
)
loaded_cart_reg = joblib.load(
    os.path.join(MODELS_DIR, "cart_decision_tree_regressor.joblib")
)
loaded_cart_clf = joblib.load(
    os.path.join(MODELS_DIR, "cart_decision_tree_classifier.joblib")
)

print("All model artifacts successfully deserialized from disk.")

# 3. Construct a synthetic scenario (e.g., Marikina high water scenario)
# Marikina: Elevation ~15.0m, WaterLevel ~16.5m, SoilMoisture ~85%
sample_input = {
    "WaterLevel_m": 16.5,
    "SoilMoisture_pct": 85.0,
    "Elevation_m": 15.0,
    "Location_Manila": 0,
    "Location_Marikina": 1,
    "Location_Pasig": 0,
    "Location_Quezon City": 0,
} 

# 4. Align inputs with exact feature signatures
X_sample_lin = pd.DataFrame(
    [[sample_input[col] for col in saved_features["linear_features"]]],
    columns=saved_features["linear_features"],
)
X_sample_cart = pd.DataFrame(
    [[sample_input[col] for col in saved_features["cart_features"]]],
    columns=saved_features["cart_features"],
)

# 5. Execute test inferences
pred_lin_val = loaded_lin_reg.predict(X_sample_lin)[0]
pred_cart_val = loaded_cart_reg.predict(X_sample_cart)[0]
pred_clf_val = loaded_cart_clf.predict(X_sample_cart)[0]
pred_clf_proba = loaded_cart_clf.predict_proba(X_sample_cart)[0][1]

print("\n--- Test Inference Results (Synthetic Scenario) ---")
print(f"Multiple Linear Regression Pred:  {pred_lin_val:.2f} mm")
print(f"CART DecisionTreeRegressor Pred:   {pred_cart_val:.2f} mm")
print(
    f"CART DecisionTreeClassifier Pred:  Class {pred_clf_val} (Flood Prob: {pred_clf_proba:.2%})"
)

All model artifacts successfully deserialized from disk.

--- Test Inference Results (Synthetic Scenario) ---
Multiple Linear Regression Pred:  98.81 mm
CART DecisionTreeRegressor Pred:   38.58 mm
CART DecisionTreeClassifier Pred:  Class 1 (Flood Prob: 97.61%)


## Step 4: Model Serialization & Export Summary

### 1. Exported Artifacts
All production-ready models and input schemas have been serialized to the `models/` directory:
- `baseline_linear_regression.joblib`: Linear baseline excluding collinear elevation.
- `cart_decision_tree_regressor.joblib`: Primary regression tree model (`max_depth=4`, `min_samples_leaf=20`).
- `baseline_logistic_regression.joblib`: Classification baseline dry run.
- `cart_decision_tree_classifier.joblib`: Primary classification tree dry run (`max_depth=4`, `min_samples_leaf=20`, `class_weight='balanced'`).
- `model_features.json`: Metadata defining strict column ordering for API and dashboard ingestion.
- **Future Note:** 
    - Re-export `cart_decision_tree_regressor.joblib` trained on actual `FloodDepth_m`.
    - Replace binary `cart_decision_tree_classifier.joblib` with a multi-class model predicting 3 MGB categories (`Low`, `Moderate`, `High`).
    - Update `model_features.json` so `Rainfall_mm` is re-added to both linear and CART feature arrays.

### 2. Inference Verification & Smoke Testing
- Re-loaded artifacts from disk and confirmed deterministic prediction capabilities across both linear and CART architectures.
- Verified that input payloads aligned against `model_features.json` execute without dimension or feature-order mismatch.
- Ready for integration with backend inference layers or dashboard components.
- **Future Note:** Update the inference smoke test scenario to assert:
    - Continuous output produces flood depth directly in meters (m).
    - Multi-class classification outputs a 3-element probability array corresponding to `[P(Low), P(Moderate), P(High)]`.